# Exercice 10 - Transformations Silver

## Objectifs
- Enrichir les donnees avec des colonnes calculees
- Appliquer des transformations metier
- Normaliser les structures de donnees
- Creer des vues denormalisees

---

## 1. Types de transformations Silver

```
+------------------------------------------------------------------+
|                    TRANSFORMATIONS SILVER                        |
+------------------------------------------------------------------+
|                                                                  |
|  +------------------------+   +------------------------+        |
|  | Colonnes calculees     |   | Enrichissement         |        |
|  +------------------------+   +------------------------+        |
|  | - Calculs arithmetiques|   | - Jointures tables     |        |
|  | - Dates derivees       |   | - Lookup externe       |        |
|  | - Categorisations      |   | - Ajout dimensions     |        |
|  +------------------------+   +------------------------+        |
|                                                                  |
|  +------------------------+   +------------------------+        |
|  | Normalisation          |   | Denormalisation        |        |
|  +------------------------+   +------------------------+        |
|  | - Types uniformes      |   | - Vue aplatie          |        |
|  | - Unites standards     |   | - Tout en une table    |        |
|  | - Codes normalises     |   | - Performance lecture  |        |
|  +------------------------+   +------------------------+        |
|                                                                  |
+------------------------------------------------------------------+
```

## 2. Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

# Creer la SparkSession
spark = SparkSession.builder \
    .appName("Transformations Silver") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0,org.apache.hadoop:hadoop-aws:3.4.1,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print("Spark pret")

In [ ]:
# Configuration PostgreSQL
jdbc_url = "jdbc:postgresql://postgres:5432/app"
jdbc_properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

# Charger les tables Northwind
df_orders = spark.read.jdbc(url=jdbc_url, table="orders", properties=jdbc_properties)
df_order_details = spark.read.jdbc(url=jdbc_url, table="order_details", properties=jdbc_properties)
df_products = spark.read.jdbc(url=jdbc_url, table="products", properties=jdbc_properties)
df_customers = spark.read.jdbc(url=jdbc_url, table="customers", properties=jdbc_properties)
df_categories = spark.read.jdbc(url=jdbc_url, table="categories", properties=jdbc_properties)

print("Tables chargees")

## 3. Colonnes calculees

In [ ]:
# Calcul du montant par ligne de commande
df_details = df_order_details.withColumn(
    "montant_ligne",
    F.round(F.col("unit_price") * F.col("quantity") * (1 - F.col("discount")), 2)
)

df_details.select("order_id", "product_id", "unit_price", "quantity", "discount", "montant_ligne").show(10)

In [ ]:
# Ajouter des colonnes de date derivees
df_orders_enrichi = df_orders \
    .withColumn("annee", F.year("order_date")) \
    .withColumn("mois", F.month("order_date")) \
    .withColumn("trimestre", F.quarter("order_date")) \
    .withColumn("jour_semaine", F.dayofweek("order_date")) \
    .withColumn("nom_jour", F.date_format("order_date", "EEEE")) \
    .withColumn("nom_mois", F.date_format("order_date", "MMMM"))

df_orders_enrichi.select("order_id", "order_date", "annee", "mois", "trimestre", "nom_jour").show(10)

In [ ]:
# Calculer le delai de livraison
df_orders_enrichi = df_orders_enrichi.withColumn(
    "delai_livraison_jours",
    F.datediff(F.col("shipped_date"), F.col("order_date"))
)

df_orders_enrichi.select("order_id", "order_date", "shipped_date", "delai_livraison_jours").show(10)

## 4. Categorisations

In [ ]:
# Categoriser les produits par niveau de prix
df_products_cat = df_products.withColumn(
    "niveau_prix",
    F.when(F.col("unit_price") < 10, "Economique")
     .when(F.col("unit_price") < 25, "Standard")
     .when(F.col("unit_price") < 50, "Premium")
     .otherwise("Luxe")
)

df_products_cat.select("product_name", "unit_price", "niveau_prix").show(10)

In [ ]:
# Categoriser les delais de livraison
df_orders_cat = df_orders_enrichi.withColumn(
    "categorie_delai",
    F.when(F.col("delai_livraison_jours").isNull(), "Non expedie")
     .when(F.col("delai_livraison_jours") <= 3, "Rapide")
     .when(F.col("delai_livraison_jours") <= 7, "Normal")
     .when(F.col("delai_livraison_jours") <= 14, "Lent")
     .otherwise("Tres lent")
)

df_orders_cat.groupBy("categorie_delai").count().show()

## 5. Enrichissement par jointure

In [ ]:
# Joindre order_details avec products
df_details_enrichi = df_details.join(
    df_products.select("product_id", "product_name", "category_id"),
    on="product_id",
    how="left"
)

df_details_enrichi.show(5)

In [ ]:
# Ajouter le nom de la categorie
df_details_enrichi = df_details_enrichi.join(
    df_categories.select("category_id", "category_name"),
    on="category_id",
    how="left"
)

df_details_enrichi.select("order_id", "product_name", "category_name", "montant_ligne").show(10)

## 6. Vue denormalisee complete

In [ ]:
# Creer une vue complete des ventes
df_ventes = df_details_enrichi.join(
    df_orders_cat.select(
        "order_id", "customer_id", "order_date", 
        "annee", "mois", "trimestre",
        "delai_livraison_jours", "categorie_delai"
    ),
    on="order_id",
    how="left"
)

# Ajouter les informations client
df_ventes = df_ventes.join(
    df_customers.select("customer_id", "company_name", "country"),
    on="customer_id",
    how="left"
)

print("Vue denormalisee des ventes :")
df_ventes.printSchema()

In [ ]:
# Afficher un apercu
df_ventes.select(
    "order_id", "order_date", "company_name", "country",
    "product_name", "category_name", "montant_ligne"
).show(10, truncate=False)

## 7. Fonctions de fenetre (Window Functions)

In [ ]:
# Classement des produits par montant dans chaque commande
window_order = Window.partitionBy("order_id").orderBy(F.desc("montant_ligne"))

df_ventes_rank = df_ventes.withColumn(
    "rang_produit",
    F.rank().over(window_order)
)

df_ventes_rank.filter(F.col("order_id") == 10248) \
    .select("order_id", "product_name", "montant_ligne", "rang_produit").show()

In [ ]:
# Calcul du pourcentage du montant par ligne
window_total = Window.partitionBy("order_id")

df_ventes_pct = df_ventes.withColumn(
    "total_commande",
    F.sum("montant_ligne").over(window_total)
).withColumn(
    "pct_commande",
    F.round(F.col("montant_ligne") / F.col("total_commande") * 100, 2)
)

df_ventes_pct.filter(F.col("order_id") == 10248) \
    .select("order_id", "product_name", "montant_ligne", "total_commande", "pct_commande").show()

In [ ]:
# Cumul des ventes par client
window_client = Window.partitionBy("customer_id").orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_cumul = df_ventes.withColumn(
    "cumul_achats",
    F.sum("montant_ligne").over(window_client)
)

df_cumul.filter(F.col("customer_id") == "VINET") \
    .select("order_id", "order_date", "montant_ligne", "cumul_achats") \
    .distinct() \
    .orderBy("order_date").show()

## 8. Fonction de transformation complete

In [ ]:
def transformer_ventes_silver(df_orders, df_order_details, df_products, df_categories, df_customers):
    """
    Cree une vue denormalisee et enrichie des ventes.
    """
    # 1. Enrichir order_details
    df = df_order_details.withColumn(
        "montant_ligne",
        F.round(F.col("unit_price") * F.col("quantity") * (1 - F.col("discount")), 2)
    )
    
    # 2. Joindre products
    df = df.join(
        df_products.select("product_id", "product_name", "category_id"),
        on="product_id",
        how="left"
    )
    
    # 3. Joindre categories
    df = df.join(
        df_categories.select("category_id", "category_name"),
        on="category_id",
        how="left"
    )
    
    # 4. Enrichir orders
    df_ord = df_orders \
        .withColumn("annee", F.year("order_date")) \
        .withColumn("mois", F.month("order_date")) \
        .withColumn("trimestre", F.quarter("order_date")) \
        .withColumn("delai_livraison_jours", F.datediff("shipped_date", "order_date"))
    
    # 5. Joindre orders
    df = df.join(
        df_ord.select("order_id", "customer_id", "order_date", "annee", "mois", "trimestre", "delai_livraison_jours"),
        on="order_id",
        how="left"
    )
    
    # 6. Joindre customers
    df = df.join(
        df_customers.select("customer_id", "company_name", "country", "city"),
        on="customer_id",
        how="left"
    )
    
    # 7. Ajouter metadata
    df = df.withColumn("_transformed_at", F.current_timestamp())
    
    return df

In [ ]:
# Appliquer la transformation
df_silver_ventes = transformer_ventes_silver(
    df_orders, df_order_details, df_products, df_categories, df_customers
)

print(f"Vue Silver : {df_silver_ventes.count()} lignes")
df_silver_ventes.show(5)

In [ ]:
# Sauvegarder dans Silver
date_traitement = datetime.now().strftime("%Y-%m-%d")
chemin_silver = f"s3a://silver/ventes/{date_traitement}"

df_silver_ventes.write.mode("overwrite").parquet(chemin_silver)
print(f"Sauvegarde : {chemin_silver}")

---

## Exercice

**Objectif** : Creer une vue Silver des employes enrichie

**Consigne** :
1. Lisez la table `employees`
2. Calculez l'anciennete (annees depuis embauche)
3. Categorisez par anciennete (Junior, Confirme, Senior, Expert)
4. Ajoutez le nombre de commandes gerees par employe
5. Sauvegardez dans Silver

A vous de jouer :

In [ ]:
# TODO: Lire employees
df_employees = spark.read.jdbc(url=jdbc_url, table="employees", properties=jdbc_properties)
df_employees.show(5)

In [ ]:
# TODO: Calculer l'anciennete
# Indice : F.datediff(F.current_date(), F.col("hire_date")) / 365

In [ ]:
# TODO: Categoriser et ajouter le nombre de commandes

---

## Resume

Dans ce notebook, vous avez appris :
- Comment creer des **colonnes calculees** (montants, dates derivees)
- Comment **categoriser** les donnees avec when/otherwise
- Comment **enrichir** les donnees par jointures
- Comment creer des **vues denormalisees**
- Comment utiliser les **fonctions de fenetre** (Window Functions)

### Prochaine etape
Dans le prochain notebook, nous apprendrons a creer des agregations pour la couche Gold.